
# STRATUM — Preprocessing Design and Leakage Prevention

## Phase 2 — Step 2.6

### Objective

Design a reproducible machine-learning preprocessing pipeline
for semiconductor virtual metrology.

### Prediction Tasks

1. Predict mean AlCu film thickness.
2. Predict mean sheet resistance.

### Data Preparation Principles

- Use APC process variables as primary model inputs.
- Include equipment metadata as an optional categorical feature.
- Exclude actual post-process measurements.
- Exclude quality labels and derived resistivity values.
- Exclude wafer identifiers and raw timestamps from initial models.
- Fit imputation and encoding using training data only.
- Preserve the original dataset.
- Do not train models or finalise features in this notebook.

### Current Status

This notebook validates the proposed data schema and
preprocessing architecture without fitting transformations
on the complete dataset.

In [1]:

# STEP 2.6.4 - Import project modules

from pathlib import Path
import sys
import pandas as pd
import numpy as np

# Determine the STRATUM project root
current_dir = Path.cwd()

if (current_dir / "ml" / "preprocessing.py").exists():
    project_root = current_dir
elif (
    current_dir.parent / "ml" / "preprocessing.py"
).exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError(
        "Cannot locate ml/preprocessing.py"
    )

# Allow Python to import our local ML module
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ml.preprocessing import (
    EXCLUDED_COLUMNS,
    TARGET_COLUMNS,
    CATEGORICAL_FEATURES,
    get_feature_columns,
    build_preprocessor,
)

print("STRATUM preprocessing module imported!")

STRATUM preprocessing module imported!


In [2]:

# STEP 2.6.5 - Load the original dataset

csv_path = (
    project_root
    / "data"
    / "raw"
    / "apc_spc_AlCu_inputs_outputs.csv"
)

df = pd.read_csv(
    csv_path,
    sep=";",
    skiprows=1,
    low_memory=False,
)

print("Dataset shape:", df.shape)

assert df.shape == (5208, 178)

print("Dataset verification successful!")

Dataset shape: (5208, 178)
Dataset verification successful!


## 1. Candidate Model Feature Validation

Identify permitted APC variables and equipment metadata.

Verify that thickness, resistance, resistivity, quality
labels, identifiers, and timestamps are excluded from
the initial model feature set.

This is a schema check, not a feature-selection exercise.

In [3]:

# STEP 2.6.6 - Validate model input features

apc_features, feature_columns = (
    get_feature_columns(df)
)

print("Numeric APC features:", len(apc_features))

print(
    "Categorical features:",
    len(CATEGORICAL_FEATURES)
)

print(
    "Total candidate input columns:",
    len(feature_columns)
)

print("\nPrediction targets:")
for target in TARGET_COLUMNS:
    print(target)

# Verify no prohibited features are present
assert not (
    set(feature_columns) & EXCLUDED_COLUMNS
    - set(CATEGORICAL_FEATURES)
)

assert not (
    set(feature_columns) & set(TARGET_COLUMNS)
)

assert feature_columns[-1] == "equipment"

print("\nFeature validation passed!")

Numeric APC features: 110
Categorical features: 1
Total candidate input columns: 111

Prediction targets:
mean_thickness
mean_resistance

Feature validation passed!


## 2. Candidate Feature Missingness

Inspect the missing-value distribution among permitted
ML input variables.

Do not calculate or apply imputation values using
the complete dataset.

In [4]:

# STEP 2.6.7 - Missingness audit

feature_missing = (
    df[feature_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(
    "Features with missing values:",
    int((feature_missing > 0).sum())
)

print(
    "Features with over 50% missingness:",
    int((feature_missing > 50).sum())
)

print(
    "Features with no missing values:",
    int((feature_missing == 0).sum())
)

display(
    feature_missing.head(20)
    .round(2)
    .to_frame("Missing Percentage")
)

Features with missing values: 47
Features with over 50% missingness: 11
Features with no missing values: 64


,Missing Percentage
WAFERCOUNT_RECIPE_MR,68.57
VOLTS_TARGET_DEP1_G,67.86
GAS2_DEP1_S,64.94
WAFERCOUNT_MR,61.54
LIFE_TARGET_RUN_I,61.12
T_RUN_ENG,54.47
PRESS_CHAMBER_DEP1_R,54.19
PRESS_TRANSFER_RUN_X,54.19
PRESS_IONGAUGE_BP_I_ENG,52.40
PRESS_IONGAUGE_BP_I_ENG1,50.56


## 3. Preprocessing Pipeline Construction

Build an unfitted preprocessing pipeline for
numerical APC features and categorical equipment data.

The preprocessing transformer must be fitted only
after the final training/test split is defined.

In [5]:

# STEP 2.6.8 - Build unfitted preprocessor

preprocessor = build_preprocessor(
    numeric_features=apc_features
)

print("Preprocessing pipeline created successfully!")

print("\nPipeline structure:")
print(preprocessor)

# Check that the transformer is not fitted
assert not hasattr(
    preprocessor,
    "transformers_"
)

print("\nConfirmed: preprocessing is not fitted.")

Preprocessing pipeline created successfully!

Pipeline structure:
ColumnTransformer(sparse_threshold=0,
                  transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                keep_empty_features=True,
                                                                strategy='median'))]),
                                 ['AMPS_TARGET_20_90_DELTA_DEP1',
                                  'AMPS_TARGET_20_90_DEP1_M',
                                  'AMPS_TARGET_20_90_DEP1_SPS',
                                  'AMPS_TARGET_20_90_DEP1_SPS_M',
                                  'AMPS_TARGET_20_90_DEP1_SPS_X',
                                  'AMPS_TARGET_20_90_DEP1_X',
                                  'AMP...
                                  'DCBIAS_PLATEN_DEP1_A_MR',
                                  'DCBIAS

## 4. Data Leakage Prevention Verification

Verify that all post-process metrology outputs,
quality labels, identifying fields, and timestamps
are excluded from the numerical APC feature set.

Equipment is the only intentionally permitted
metadata feature in the initial configuration.

In [6]:

# STEP 2.6.9 - Leakage prevention checks

prohibited = sorted(
    set(feature_columns) & (
        EXCLUDED_COLUMNS - {"equipment"}
    )
)

assert len(prohibited) == 0, (
    f"Potential leakage features detected: {prohibited}"
)

assert set(apc_features).isdisjoint(EXCLUDED_COLUMNS)

assert set(TARGET_COLUMNS).isdisjoint(feature_columns)

leakage_report = pd.DataFrame({
    "Check": [
        "APC sensor count",
        "Permitted equipment feature",
        "Post-metrology leakage features",
        "Target leakage features",
        "Preprocessor fitted on full dataset"
    ],
    "Result": [
        len(apc_features),
        "equipment" in feature_columns,
        len(prohibited),
        len(set(TARGET_COLUMNS) & set(feature_columns)),
        hasattr(preprocessor, "transformers_")
    ]
})

display(leakage_report)

print("\nSTRATUM leakage validation passed!")

,Check,Result
0,APC sensor count,110
1,Permitted equipment feature,True
2,Post-metrology leakage features,0
3,Target leakage features,0
4,Preprocessor fitted on full dataset,False



STRATUM leakage validation passed!



## 5. Preprocessing Strategy and Design Decisions

### Primary Prediction Tasks

STRATUM will evaluate two virtual metrology regression tasks:

1. Prediction of mean AlCu film thickness.
2. Prediction of mean AlCu sheet resistance.

Both tasks use process measurements recorded during
manufacturing, rather than post-process metrology outputs.

### Candidate Features

- 110 numeric Advanced Process Control variables.
- Equipment identifier as one optional categorical input.
- Machine identifier excluded initially to avoid unnecessary
  redundancy with equipment metadata.

### Excluded Variables

- All 17 measured film-thickness values.
- All 17 measured sheet-resistance values.
- All 17 derived resistivity values.
- Recorded mean thickness and mean resistance.
- Individual and overall quality indicators.
- stable_state, which is derived from a process power criterion.
- Wafer, lot and batch identifiers.
- Raw production timestamps and wafer slot.

### Missing-Value Strategy

Initial baseline:
- Median imputation for numerical APC measurements.
- Missing-value indicators for numeric features.
- Most-frequent imputation for equipment metadata.

Important:
- Imputation must be fitted on training data only.
- Missingness patterns may differ across equipment.
- High-missingness features are not automatically removed.
- Alternative strategies may be evaluated if justified.

### Categorical Encoding

- One-hot encoding for equipment.
- Unknown equipment categories handled without errors.
- Compare APC-only and APC-plus-equipment configurations
  during model development.

### Feature Redundancy

Previous EDA identified highly correlated APC sensors.

Correlated features will not be automatically removed.

Feature selection, if used, must be performed within the
training process and evaluated using held-out data.

### Outlier Handling

Previous EDA identified unusual sensor values.

They will not be automatically deleted or clipped.

Outlier treatment will be considered using training-only
analysis and evaluated for its effects on model performance.

### Data Splitting

The preliminary chronological 80/20 split showed:
- 4,166 earlier wafer records.
- 1,042 later wafer records.
- No production-lot overlap across partitions.

The final split will be established in Step 2.7.

### Data Leakage Prevention

- Never train using post-process measurement outputs as inputs.
- Never fit imputers or encoders on held-out test data.
- Never select features using test-set target correlations.
- Keep related production lots from leaking across partitions.
- Preserve the chronological ordering of the final evaluation.
- Use training-only transformations during cross-validation.

### Current Status

The preprocessing module has been implemented.

The candidate features have been identified and validated.

An unfitted preprocessing pipeline has been constructed.

No model training, final feature selection, imputation fitting,
or permanent data transformation has occurred.

### Next Step

Finalize chronological and lot-aware dataset splitting,
fit preprocessing on training data, and prepare reproducible
inputs for subsequent virtual metrology experiments.